# Company Hierarchy
**Company:** Atlassian (GothamLoop question bank) · **Category:** Coding · **Tags:** Onsite Loop, Trees · **Difficulty/Frequency:** Rare (2/10)


## Concepts

**What this problem is really testing:**
- Lowest common ancestor (LCA) — but for a whole *group* of target nodes, not just two
- Solved with one post-order DFS that counts, per subtree, how many targets it contains

**Why it applies here:**
- "The lowest common manager of {Alice, Bob, ...}" is really just "the deepest node whose subtree contains *every one* of these people" — the standard LCA idea, just generalized from 2 targets to k.
- A post-order traversal fits naturally: a node can only know "how many targets are underneath me" once all of its children have already reported their own counts.

**The one idea to hold onto:** don't build and compare full root-to-target paths for every target (that costs O(k·h) time and space for k targets). Instead, have every subtree report *how many* targets it contains. The first node — found while working bottom-up — whose count reaches the full target count is, by construction, the lowest node containing all of them.

---

### Quick primers — the building blocks used below

**Post-order (bottom-up) tree aggregation.**
- A traversal that fully processes all of a node's children **before** combining their results into that node's own answer.
- Mental model: `answer(node) = combine(own_contribution(node), answer(child_1), answer(child_2), ...)`.
- Every node is visited once, so total cost is O(total nodes) times whatever `combine` costs per node.

**What is a Hash Set?**
- A hash set (Python `set`) stores unique elements, giving **O(1) average** membership testing, insertion, and deletion — same underlying trick as a hash map, just without attached values.
- Used here to turn "is this employee one of the targets?" into a quick O(1) check, instead of scanning the whole target list (O(k)) every time.

**Lowest Common Ancestor (LCA), generalized to k targets.**
- For 2 targets: LCA is "the deepest node that is an ancestor of both".
- For k targets: it generalizes to "the deepest node whose subtree contains *all* of them".
- The count-based post-order DFS below computes exactly that, for any k, in a single traversal.


## Problem Statement

Given the root of an M-ary tree (company hierarchy: root=company, internal nodes=departments/teams, leaves=employees; all names unique) and a list of employee names, return the name of the lowest common manager for all of them.

**Example** (tree: Company -> {Sales -> {Alice, Bob}, Tech -> {Carl, Dan}, HR -> {Eva}})
- `nodes=["Alice", "Bob"]` -> `"Sales"` (both under Sales).
- `nodes=["Alice", "Dan"]` -> `"Company"` (Alice is under Sales, Dan under Tech -- only Company contains both).


### Approach 1 -- Naive (build every root-to-target path, then compare)

**Idea:** for each of the k target names, walk the tree to build its full root-to-node path. Then find the deepest node common to all k paths -- e.g. by walking the paths in lockstep from the root until they diverge.

**Time complexity:** each per-target search is a DFS from the root that, in the worst case, has to fully explore other subtrees before it stumbles onto the target -- up to O(n) nodes visited for a single target, not just O(h) (path length). So the total is **O(k * n)** worst case for k targets, dominating the O(k * h) lockstep comparison at the end. (If you instead had a name -> node index built once, each lookup drops to O(1) and only the O(h) walk up to the root would remain per target -- but that's already most of the way to a different algorithm.)

**Space complexity:** O(k * h) to store all k paths simultaneously.


In [ ]:
from typing import List, Optional


class Node:
    def __init__(self, name: str, children: Optional[List["Node"]] = None):
        self.name = name
        self.children = children if children is not None else []


def _find_path(node: Optional[Node], target: str, path: List[str]) -> bool:
    """DFS that records the path from `node` down to `target`, if found."""
    if node is None:
        return False
    path.append(node.name)
    if node.name == target:
        return True
    for child in node.children:
        if _find_path(child, target, path):
            return True
    path.pop()          # backtrack: this branch doesn't contain the target
    return False


def lowest_common_manager_naive(root: Node, target_names: List[str]) -> Optional[str]:
    paths = []
    for name in target_names:
        path: List[str] = []
        _find_path(root, name, path)          # each call re-walks from the root
        paths.append(path)

    if not paths or any(not p for p in paths):
        return None

    lca = None
    for level in zip(*paths):                  # walk all paths in lockstep
        if len(set(level)) == 1:                # every path agrees at this depth
            lca = level[0]
        else:
            break
    return lca


### Approach 2 -- Optimal (single post-order DFS, count targets per subtree)

**Idea:** one DFS. Each call returns how many of the target names appear in its subtree (including the node itself). The **first** node (encountered bottom-up, i.e. in post-order) whose count equals the total number of targets is the answer -- everything above it will also have the full count, but it won't be the *lowest* such node.

**Time complexity:** O(n) -- every node is visited exactly once, and each membership check against the target set is O(1) average.

**Space complexity:** O(h) for the recursion stack (h = tree height, worst case O(n)).


In [ ]:
def lowest_common_manager(root: Node, target_names: List[str]) -> Optional[str]:
    targets = set(target_names)
    total_targets = len(targets)
    answer: Optional[Node] = None

    def dfs(node: Optional[Node]) -> int:
        nonlocal answer
        if node is None:
            return 0

        count = 1 if node.name in targets else 0
        for child in node.children:
            count += dfs(child)                 # children finish (post-order) before we combine

        if count == total_targets and answer is None:   # first (=lowest) node to reach the full count
            answer = node

        return count

    dfs(root)
    return answer.name if answer else None


## Verification

Build the example tree and confirm both approaches agree, plus the edge cases the Talking Points call out.

In [ ]:
company = Node("Company", [
    Node("Sales", [Node("Alice"), Node("Bob")]),
    Node("Tech", [Node("Carl"), Node("Dan")]),
    Node("HR", [Node("Eva")]),
])

for fn in (lowest_common_manager_naive, lowest_common_manager):
    assert fn(company, ["Alice", "Bob"]) == "Sales", fn.__name__
    assert fn(company, ["Alice", "Dan"]) == "Company", fn.__name__
    assert fn(company, ["Alice"]) == "Alice", fn.__name__          # a single target IS its own LCA
    assert fn(company, ["Carl", "Dan"]) == "Tech", fn.__name__
    assert fn(company, ["Alice", "Bob", "Eva"]) == "Company", fn.__name__

# Ancestor edge case: one target IS the manager of another
assert lowest_common_manager(company, ["Sales", "Alice"]) == "Sales"     # Sales' own subtree already has both
assert lowest_common_manager_naive(company, ["Sales", "Alice"]) == "Sales"

# All employees -> root
all_names = ["Alice", "Bob", "Carl", "Dan", "Eva"]
assert lowest_common_manager(company, all_names) == "Company"

# Deeper hierarchy: nested departments
deep = Node("Company", [
    Node("Eng", [
        Node("Backend", [Node("Priya"), Node("Sam")]),
        Node("Frontend", [Node("Wei")]),
    ]),
])
assert lowest_common_manager(deep, ["Priya", "Sam"]) == "Backend"
assert lowest_common_manager(deep, ["Priya", "Wei"]) == "Eng"

print("All checks passed.")


## Discussion -- remaining follow-up directions

- **Extremely deep trees risking stack overflow.** Convert to an iterative post-order traversal with an explicit stack -- the same "process children before combining" logic, just without relying on the call stack (mirrors the iterative traversal shown in the Confluence Page Word Count notebook for the same underlying reason).
- **Multiple LCA queries on the same tree.** Preprocess once with **binary lifting** (O(n log n) preprocessing, O(log n) per pairwise query) or an **Euler tour + sparse-table RMQ** (O(n log n) preprocessing, O(1) per pairwise query) -- both extend the 2-target case; generalizing either to a k-target query per call needs more thought (e.g. reduce k targets to k-1 pairwise LCA calls and take the "highest" result).
- **A tree that changes over time (employees join/leave).** For occasional changes, simply re-running the O(n) DFS is often fine. For frequent changes at scale, look at dynamic/link-cut tree structures that support LCA under edge insertions and deletions in polylog time -- meaningfully more complex, worth naming as "possible but heavy machinery" rather than implementing live.
- **Duplicate names (uniqueness constraint removed).** Switch from name-based `set` membership to identity-based tracking -- pass target **node references** instead of name strings, and compare with `is`/`id()` instead of `==`/hashing on name.
- **Targets given as node references instead of names.** The solution gets slightly simpler: no name->target lookup needed, and identity comparison (`node in target_set`) sidesteps any naming ambiguity entirely -- shown below.


In [ ]:
def lowest_common_manager_by_ref(root: Node, target_nodes: List[Node]) -> Optional[Node]:
    """Same algorithm, but keyed by node identity instead of name -- sidesteps duplicate names."""
    targets = set(id(n) for n in target_nodes)
    total_targets = len(targets)
    answer: Optional[Node] = None

    def dfs(node: Optional[Node]) -> int:
        nonlocal answer
        if node is None:
            return 0
        count = 1 if id(node) in targets else 0
        for child in node.children:
            count += dfs(child)
        if count == total_targets and answer is None:
            answer = node
        return count

    dfs(root)
    return answer


alice_node = company.children[0].children[0]   # Sales -> Alice
bob_node = company.children[0].children[1]      # Sales -> Bob
result = lowest_common_manager_by_ref(company, [alice_node, bob_node])
assert result is company.children[0]            # the Sales Node object itself
print("Reference-based LCA works:", result.name)


## Empirical complexity check

`lowest_common_manager` (optimal) is O(n): one fixed-cost pass over the whole tree, **independent of k** -- so its time should barely move as k (the number of targets queried) grows.

The naive approach is more interesting to actually measure than to guess: its per-target `_find_path` walks children **in order** and only stops once it stumbles onto the target, so a target sitting in the d-th department forces a full scan of departments `0..d-1` first. Our benchmark queries targets `emp0_0, emp1_0, ..., emp(k-1)_0` -- so as k grows, both *the number of searches* and *each search's own cost* grow with k, compounding to **quadratic** total work, even though the tree itself (n = 20,000) never changes size. This is a good real-world lesson: a loose worst-case bound like O(k*n) can hide a much worse *actual* growth rate for a specific, realistic access pattern.

| Growth when k doubles | Implies |
|---|---|
| ~1x | optimal DFS -- flat, independent of k |
| ~4x | naive -- quadratic in k for this ordered-target access pattern |


In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

# A wide, shallow tree: 1 root, 200 departments, 100 employees each -- 20,000 employees, height 2.
DEPTS = 200
EMPLOYEES_PER_DEPT = 100
WIDE_TREE = Node("Company", [
    Node(f"dept{d}", [Node(f"emp{d}_{e}") for e in range(EMPLOYEES_PER_DEPT)])
    for d in range(DEPTS)
])
ALL_TARGETS = [f"emp{d}_{0}" for d in range(DEPTS)]   # one target per department -> LCA is the root


def make_worst_case(k):
    return (WIDE_TREE, ALL_TARGETS[:k])


solutions = {
    "naive (per-target path search)": lowest_common_manager_naive,
    "optimal (single count DFS)": lowest_common_manager,
}
sizes = [20, 40, 80, 160]     # number of targets k
benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Post-order "count/aggregate per subtree, answer is the first node hitting the target" is the general LCA-for-a-set pattern.** It generalizes cleanly from 2 targets to k with no structural change -- only the "total_targets" threshold changes.
- **The `answer is None` guard is what makes it "lowest", not just "an ancestor".** Post-order visits children before parents, so the *first* node whose count reaches the threshold is provably the deepest (lowest) one; without the guard, every ancestor above it would overwrite the answer.
- **A hash set turns "is this a target?" into O(1)**, keeping the whole traversal O(n) instead of O(n*k) from a linear scan of the target list at every node.
- **Space-time tradeoff is worth stating explicitly.** Path-building is O(k*h) space; the count-based DFS is O(h) regardless of k -- naming this trade-off is itself a talking point, not just an implementation detail.
- **Related problems:** LCA of two nodes in a binary tree (LeetCode 236, the k=2 special case), LCA of a BST (uses the ordering property instead of a full traversal), Euler tour + RMQ for O(1) repeated LCA queries.
- **Common pitfalls:** forgetting the `answer is None` guard (returns the *root* instead of the true LCA, since every ancestor also satisfies the count check); treating "single target" as a special case needing different code (it isn't -- the count-based approach handles k=1 naturally, the target itself becomes its own LCA); assuming names are unique when the follow-up removes that constraint.
